# Recomendar es factorizar

**Matemática 4 — Facultad de Informática, UNLP**

Este notebook implementa la factorización $R \approx U V^\top$ de la matriz usuario–película de
**MovieLens latest-small** con dos métodos:

- **ALS** (mínimos cuadrados alternados), que resuelve cada fila con las ecuaciones normales;
- **descenso de gradiente** completo sobre la suma de cuadrados del error.

Todo el álgebra está escrita con numpy, sin librerías que factoricen o recomienden.

**Cómo correrlo:** *Entorno de ejecución → Ejecutar todo*. La primera vez descarga MovieLens (unos 1 MB)
y el notebook completo tarda menos de un minuto.

## 1. Instalación

Colab ya trae numpy y matplotlib; la celda solo se asegura de que estén. No hace falta nada más.

In [ ]:
!pip install -q numpy matplotlib

## 2. Importaciones y parámetros

Los hiperparámetros quedan todos juntos en esta celda. Se pueden cambiar y volver a ejecutar el
notebook desde acá.

| Parámetro | Valor | Por qué |
|---|---|---|
| `UMBRAL` | 40 | Mínimo de calificaciones por usuario y por película. Es el menor umbral con el que ALS y descenso de gradiente dan estimaciones razonables fuera de Ω (máximo $\lvert\hat r\rvert$ menor a 7). |
| `K` | 2 | Dimensión latente. Con k = 3 o 5, sin regularización, ALS da estimaciones absurdas fuera de Ω. |
| `ETA` | 5e-4 | Tasa de aprendizaje. Con 1e-3 el descenso diverge; con 2e-4 tarda el doble. |
| `EPSILON` | 1.0 | Corte: $\lvert f(t+1) - f(t)\rvert < \varepsilon$. La SCE está en el orden de las decenas de miles. |
| `MAX_ITER` | 3000 | Tope de iteraciones; el descenso de gradiente corta antes, en unas 460. |
| `SEMILLA` | 42 | Solo para que los resultados sean reproducibles. |

In [ ]:
import csv
import logging
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
LOGGER = logging.getLogger("recomendar")

# Preprocesamiento y modelo
UMBRAL = 40
K = 2
ETA = 5e-4
EPSILON = 1.0
MAX_ITER = 3000
SEMILLA = 42
ESCALA_INICIALIZACION = 1.0  # U0 y V0 uniformes en [0, 1)

# Recomendaciones
USUARIO = 1     # id de usuario tal como aparece en MovieLens
TOP_N = 10

# Datos
URL_MOVIELENS = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
DIRECTORIO_DATOS = Path("data")

## 3. Errores propios

Excepciones con mensajes que dicen qué falló y dónde (por ejemplo, qué fila de ALS dio un sistema
singular), en lugar de un error genérico de numpy.

In [ ]:
class ErrorFactorizacion(Exception):
    "Excepción base del notebook."


class ErrorDescargaDataset(ErrorFactorizacion):
    def __init__(self, url: str) -> None:
        super().__init__(
            f"no se pudo descargar MovieLens desde {url}. Descargalo a mano desde esa URL, "
            "subí el .zip a Colab y descomprimilo en data/."
        )


class UmbralInsuficienteError(ErrorFactorizacion):
    def __init__(self, umbral: int, k: int) -> None:
        super().__init__(
            f"el umbral del filtro ({umbral}) es menor que k ({k}): ALS necesita al menos "
            f"k={k} observaciones por fila para que el sistema k×k tenga solución única."
        )


class ErrorDatosInsuficientes(ErrorFactorizacion):
    def __init__(self, umbral: int) -> None:
        super().__init__(
            f"el filtro por umbral={umbral} eliminó todos los usuarios o todas las películas."
        )


class SistemaSingularError(ErrorFactorizacion):
    def __init__(self, fila: int, n_observados: int) -> None:
        super().__init__(
            f"las ecuaciones normales de la fila {fila} son singulares: "
            f"solo hay {n_observados} observaciones para resolver el sistema."
        )


class DivergenciaError(ErrorFactorizacion):
    def __init__(self, iteracion: int, eta: float) -> None:
        super().__init__(
            f"el descenso de gradiente diverge en la iteración {iteracion}: f dejó de ser "
            f"finito con eta={eta:.4g}; probá con un eta más chico."
        )


class UsuarioNoEncontradoError(ErrorFactorizacion):
    def __init__(self, usuario_id: int) -> None:
        super().__init__(
            f"el usuario {usuario_id} no está en el conjunto filtrado (no existe en MovieLens "
            "o el filtro por UMBRAL lo eliminó): probá con otro USUARIO o con un UMBRAL más chico."
        )


class CalificacionesNuevasError(ErrorFactorizacion):
    def __init__(self, detalle: str) -> None:
        super().__init__(f"calificaciones del usuario nuevo: {detalle}")

## 4. Descarga de MovieLens

El dataset no se incluye en el repositorio porque la licencia de GroupLens no permite
redistribuirlo. Se descarga del sitio oficial y se descomprime en `data/ml-latest-small/`.
Si la carpeta ya existe, no se vuelve a descargar.

In [ ]:
def descargar_movielens(directorio_destino: Path, url: str = URL_MOVIELENS) -> Path:
    "Descarga MovieLens latest-small en `directorio_destino` y lo descomprime."
    directorio_dataset = directorio_destino / "ml-latest-small"
    if directorio_dataset.exists():
        LOGGER.info("MovieLens ya está en %s, no se descarga de nuevo", directorio_dataset)
        return directorio_dataset

    directorio_destino.mkdir(parents=True, exist_ok=True)
    ruta_zip = directorio_destino / "ml-latest-small.zip"
    LOGGER.info("descargando MovieLens desde %s", url)
    try:
        urlretrieve(url, ruta_zip)
    except (URLError, OSError) as error:
        raise ErrorDescargaDataset(url) from error

    with zipfile.ZipFile(ruta_zip) as archivo_zip:
        archivo_zip.extractall(directorio_destino)
    ruta_zip.unlink()
    return directorio_dataset


RUTA_MOVIELENS = descargar_movielens(DIRECTORIO_DATOS)
print(sorted(p.name for p in RUTA_MOVIELENS.iterdir()))

## 5. La matriz R y la máscara M

`ratings.csv` tiene una calificación por línea (de 0,5 a 5). Con eso se arman:

- $R$, de $m \times n$: la calificación del usuario $i$ a la película $j$, con `np.nan` en los huecos.
  Los huecos **nunca** se rellenan con ceros ni con promedios.
- $M$, la máscara de $\Omega$: `M[i, j] = True` si el usuario $i$ calificó la película $j$.

Los ids de MovieLens se reindexan desde 0 para usarlos como índices de fila y columna.

In [ ]:
@dataclass
class ConjuntoCalificaciones:
    "R (con NaN en los huecos), M y los mapeos de id de MovieLens a índice."
    R: np.ndarray
    M: np.ndarray
    id_usuario_a_indice: dict[int, int]
    id_pelicula_a_indice: dict[int, int]


def cargar_calificaciones(ruta_archivo: Path) -> ConjuntoCalificaciones:
    "Lee ratings.csv y arma R y M, reindexando usuarios y películas desde 0."
    filas: list[tuple[int, int, float]] = []
    with ruta_archivo.open(encoding="utf-8", newline="") as archivo:
        for registro in csv.DictReader(archivo):
            filas.append((int(registro["userId"]), int(registro["movieId"]), float(registro["rating"])))

    ids_usuario = sorted({u for u, _, _ in filas})
    ids_pelicula = sorted({p for _, p, _ in filas})
    id_usuario_a_indice = {id_crudo: i for i, id_crudo in enumerate(ids_usuario)}
    id_pelicula_a_indice = {id_crudo: j for j, id_crudo in enumerate(ids_pelicula)}

    R = np.full((len(ids_usuario), len(ids_pelicula)), np.nan)
    M = np.zeros(R.shape, dtype=bool)
    for id_usuario, id_pelicula, calificacion in filas:
        i = id_usuario_a_indice[id_usuario]
        j = id_pelicula_a_indice[id_pelicula]
        R[i, j] = calificacion
        M[i, j] = True

    LOGGER.info("cargadas %d calificaciones de %d usuarios y %d películas",
                len(filas), R.shape[0], R.shape[1])
    return ConjuntoCalificaciones(R, M, id_usuario_a_indice, id_pelicula_a_indice)

### Filtro por cantidad mínima de calificaciones

Se eliminan los usuarios y las películas con menos de `UMBRAL` calificaciones. Hay que repetirlo
hasta que no cambie nada: sacar una película puede dejar a un usuario por debajo del umbral, y al
revés.

El umbral tiene que ser al menos $k$: en ALS cada fila resuelve un sistema $k \times k$, que necesita
al menos $k$ observaciones para tener solución única. Con exactamente $k$ la solución existe pero no
es razonable, por eso el umbral que usamos es bastante mayor.

In [ ]:
def filtrar_por_minimo(datos: ConjuntoCalificaciones, umbral: int, k: int) -> ConjuntoCalificaciones:
    "Elimina filas y columnas con menos de `umbral` calificaciones, en cascada."
    if umbral < k:
        raise UmbralInsuficienteError(umbral, k)

    filas = np.arange(datos.M.shape[0])
    columnas = np.arange(datos.M.shape[1])
    while True:
        submascara = datos.M[np.ix_(filas, columnas)]
        filas_nuevas = filas[submascara.sum(axis=1) >= umbral]
        columnas_nuevas = columnas[submascara.sum(axis=0) >= umbral]
        if len(filas_nuevas) == len(filas) and len(columnas_nuevas) == len(columnas):
            break
        if len(filas_nuevas) == 0 or len(columnas_nuevas) == 0:
            raise ErrorDatosInsuficientes(umbral)
        filas, columnas = filas_nuevas, columnas_nuevas

    indice_a_id_usuario = {i: id_crudo for id_crudo, i in datos.id_usuario_a_indice.items()}
    indice_a_id_pelicula = {j: id_crudo for id_crudo, j in datos.id_pelicula_a_indice.items()}
    M_filtrada = datos.M[np.ix_(filas, columnas)]

    LOGGER.warning("filtro por umbral=%d: se eliminaron %d usuarios, %d películas y %d calificaciones",
                   umbral, datos.M.shape[0] - len(filas), datos.M.shape[1] - len(columnas),
                   int(datos.M.sum() - M_filtrada.sum()))

    return ConjuntoCalificaciones(
        R=datos.R[np.ix_(filas, columnas)],
        M=M_filtrada,
        id_usuario_a_indice={indice_a_id_usuario[viejo]: nuevo for nuevo, viejo in enumerate(filas)},
        id_pelicula_a_indice={indice_a_id_pelicula[viejo]: nuevo for nuevo, viejo in enumerate(columnas)},
    )


def cargar_titulos(ruta_movies: Path) -> dict[int, str]:
    "Lee movies.csv: id de película de MovieLens → título."
    with ruta_movies.open(encoding="utf-8", newline="") as archivo:
        return {int(registro["movieId"]): registro["title"] for registro in csv.DictReader(archivo)}

In [ ]:
datos = filtrar_por_minimo(cargar_calificaciones(RUTA_MOVIELENS / "ratings.csv"), UMBRAL, K)
titulos_por_id = cargar_titulos(RUTA_MOVIELENS / "movies.csv")
titulos_por_indice = {j: titulos_por_id[id_pelicula] for id_pelicula, j in datos.id_pelicula_a_indice.items()}

R, M = datos.R, datos.M
m, n = R.shape
print(f"MovieLens filtrado (umbral={UMBRAL}): {m} usuarios, {n} películas, {int(M.sum())} calificaciones.")
print(f"Densidad de R: {M.mean():.1%} de las posiciones están en Ω.")

## 6. Lo que comparten los dos métodos

ALS y descenso de gradiente comparten solamente la predicción, la función a minimizar y la
inicialización. La lógica de actualización de cada uno está en su propia sección.

**Predicción**: $\hat R = U V^\top$, con $U \in \mathbb{R}^{m \times k}$ y $V \in \mathbb{R}^{n \times k}$.

**Función a minimizar**: la suma de cuadrados del error, **solo sobre $\Omega$**,

$$f(U, V) = \sum_{(i,j) \in \Omega} \left(r_{ij} - u_i \cdot v_j\right)^2 .$$

**Inicialización**: $U_0$ y $V_0$ con entradas uniformes en $[0, 1)$, generadas una sola
vez con la semilla, para que los dos métodos arranquen del mismo punto.

In [ ]:
def predecir(U: np.ndarray, V: np.ndarray) -> np.ndarray:
    "R̂ = U·Vᵀ."
    return U @ V.T


def sce(R: np.ndarray, M: np.ndarray, U: np.ndarray, V: np.ndarray) -> float:
    "Suma de cuadrados del error sobre Ω. Fuera de Ω no cuenta."
    error = np.where(M, R - predecir(U, V), 0.0)
    return float(np.sum(error**2))


def inicializar_factores(m: int, n: int, k: int, escala: float,
                         generador: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    "U0 (m×k) y V0 (n×k) uniformes en [0, escala)."
    return generador.uniform(0.0, escala, size=(m, k)), generador.uniform(0.0, escala, size=(n, k))


@dataclass
class ResultadoEntrenamiento:
    "Factores finales, valores de f por iteración (incluido f0), iteraciones, tiempo y motivo de corte."
    U: np.ndarray
    V: np.ndarray
    historial_f: list[float]
    n_iteraciones: int
    tiempo_segundos: float
    motivo_corte: str

## 7. ALS: mínimos cuadrados alternados

Con $V$ fija, $f$ se separa en un problema de mínimos cuadrados independiente para cada usuario.
Si $V_{\Omega_i}$ son las filas de $V$ de las películas que calificó el usuario $i$ y $r_{\Omega_i}$ sus
calificaciones, la fila $u_i$ sale de las **ecuaciones normales**:

$$\left(V_{\Omega_i}^\top V_{\Omega_i}\right) u_i = V_{\Omega_i}^\top r_{\Omega_i} .$$

El paso de $V$ es el mismo problema mirado desde las películas: es la **misma función**, llamada con
$R^\top$, $M^\top$ y $U$. Cada sistema se resuelve con `np.linalg.solve`.

ALS alterna los dos pasos hasta que $\lvert f(t+1) - f(t)\rvert < \varepsilon$.

In [ ]:
def resolver_factor(R: np.ndarray, M: np.ndarray, F: np.ndarray) -> np.ndarray:
    "Resuelve por fila las ecuaciones normales. Con (R, M, V) da U; con (R.T, M.T, U) da V."
    filas, k = R.shape[0], F.shape[1]
    factor_nuevo = np.empty((filas, k))
    for i in range(filas):
        columnas_obs = np.where(M[i, :])[0]
        F_obs = F[columnas_obs, :]
        r_obs = R[i, columnas_obs]

        # Ecuaciones normales de mínimos cuadrados de la fila i
        A = F_obs.T @ F_obs
        b = F_obs.T @ r_obs
        try:
            factor_nuevo[i, :] = np.linalg.solve(A, b)
        except np.linalg.LinAlgError as error:
            raise SistemaSingularError(fila=i, n_observados=len(columnas_obs)) from error
    return factor_nuevo


def entrenar_als(R: np.ndarray, M: np.ndarray, U0: np.ndarray, V0: np.ndarray,
                 epsilon: float, max_iter: int) -> ResultadoEntrenamiento:
    "Alterna el paso de U y el de V hasta |f(t+1) − f(t)| < epsilon o max_iter."
    inicio = time.perf_counter()
    U, V = U0, V0
    historial_f = [sce(R, M, U, V)]
    motivo_corte = "max_iter"

    for _ in range(max_iter):
        # Paso de U y paso de V, con la misma función (relación inversa)
        U = resolver_factor(R, M, V)
        V = resolver_factor(R.T, M.T, U)
        historial_f.append(sce(R, M, U, V))
        if abs(historial_f[-1] - historial_f[-2]) < epsilon:
            motivo_corte = "tolerancia"
            break

    if motivo_corte == "max_iter":
        LOGGER.warning("ALS: se alcanzó max_iter=%d sin cortar por tolerancia", max_iter)
    n_iteraciones = len(historial_f) - 1
    LOGGER.info("ALS terminó: %d iteraciones (%s), f final=%.6g", n_iteraciones, motivo_corte, historial_f[-1])
    return ResultadoEntrenamiento(U, V, historial_f, n_iteraciones, time.perf_counter() - inicio, motivo_corte)

## 8. Descenso de gradiente

Con $E = M \odot (R - U V^\top)$, el error con ceros fuera de $\Omega$, los gradientes de $f$ son:

$$\nabla_U f = -2\, E\, V, \qquad \nabla_V f = -2\, E^\top U .$$

Es la versión **completa** (no estocástica): en cada iteración se usan todas las calificaciones.
Los dos gradientes se calculan con $U$ y $V$ de la iteración $t$ y se actualizan **juntos**:

$$U^{(t+1)} = U^{(t)} - \eta\, \nabla_U f\big(U^{(t)}, V^{(t)}\big), \qquad
V^{(t+1)} = V^{(t)} - \eta\, \nabla_V f\big(U^{(t)}, V^{(t)}\big).$$

Si se actualizara $U$ y después se calculara el gradiente de $V$ con la $U$ nueva, el método pasaría
a ser alternado, como ALS.

In [ ]:
def gradiente_sce(R: np.ndarray, M: np.ndarray, U: np.ndarray,
                  V: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    "∇_U f y ∇_V f de la SCE, los dos evaluados en el mismo (U, V)."
    E = np.where(M, R - predecir(U, V), 0.0)
    return -2 * E @ V, -2 * E.T @ U


def entrenar_gd(R: np.ndarray, M: np.ndarray, U0: np.ndarray, V0: np.ndarray,
                eta: float, epsilon: float, max_iter: int) -> ResultadoEntrenamiento:
    "Descenso de gradiente completo con actualización simultánea de U y V."
    inicio = time.perf_counter()
    U, V = U0, V0
    historial_f = [sce(R, M, U, V)]
    motivo_corte = "max_iter"

    for iteracion in range(1, max_iter + 1):
        grad_U, grad_V = gradiente_sce(R, M, U, V)
        # x(t+1) = x(t) − eta·∇f(x(t)), U y V con el gradiente de la iteración t
        U = U - eta * grad_U
        V = V - eta * grad_V

        f_siguiente = sce(R, M, U, V)
        if not np.isfinite(f_siguiente):
            raise DivergenciaError(iteracion, eta)
        if f_siguiente > historial_f[-1]:
            LOGGER.warning("GD: f aumentó en la iteración %d; eta=%.4g podría ser grande", iteracion, eta)
        historial_f.append(f_siguiente)
        if iteracion % 100 == 0:
            LOGGER.info("GD iteración %d: f=%.6g", iteracion, f_siguiente)
        if abs(historial_f[-1] - historial_f[-2]) < epsilon:
            motivo_corte = "tolerancia"
            break

    if motivo_corte == "max_iter":
        LOGGER.warning("GD: se alcanzó max_iter=%d sin cortar por tolerancia", max_iter)
    n_iteraciones = len(historial_f) - 1
    LOGGER.info("GD terminó: %d iteraciones (%s), f final=%.6g", n_iteraciones, motivo_corte, historial_f[-1])
    return ResultadoEntrenamiento(U, V, historial_f, n_iteraciones, time.perf_counter() - inicio, motivo_corte)

## 9. Entrenamiento

Se genera **un solo** par $U_0, V_0$ y cada método recibe su propia copia, así la comparación arranca
del mismo punto.

In [ ]:
generador = np.random.default_rng(SEMILLA)
U0, V0 = inicializar_factores(m, n, K, ESCALA_INICIALIZACION, generador)

resultado_als = entrenar_als(R, M, U0.copy(), V0.copy(), epsilon=EPSILON, max_iter=MAX_ITER)
resultado_gd = entrenar_gd(R, M, U0.copy(), V0.copy(), eta=ETA, epsilon=EPSILON, max_iter=MAX_ITER)

## 10. Comparación

Para cada método: cuántas iteraciones hizo, por qué cortó, cuánto tardó y el valor final de $f$.

In [ ]:
print(f"{'método':<10}{'iteraciones':>12}{'motivo de corte':>18}{'tiempo (s)':>12}{'SCE final':>14}")
for nombre, resultado in [("ALS", resultado_als), ("GD", resultado_gd)]:
    print(f"{nombre:<10}{resultado.n_iteraciones:>12}{resultado.motivo_corte:>18}"
          f"{resultado.tiempo_segundos:>12.4f}{resultado.historial_f[-1]:>14.4f}")

### Convergencia

$f$ en función de la iteración, en escala logarítmica. ALS baja casi todo en las primeras iteraciones,
porque cada paso resuelve un problema de mínimos cuadrados de forma exacta. El descenso de gradiente
avanza de a pasos chicos de tamaño $\eta$ y necesita muchas más iteraciones, aunque cada una es
más barata.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(resultado_als.historial_f, label="ALS")
ax.plot(resultado_gd.historial_f, label="Descenso de gradiente")
ax.set_yscale("log")
ax.set_xlabel("iteración")
ax.set_ylabel("f (SCE)")
ax.set_title("Convergencia sobre MovieLens filtrado")
ax.legend()
plt.show()

## 11. Recomendaciones

Para un usuario $i$, la estimación de cada película es $\hat r_{ij} = u_i \cdot v_j$. Se recomiendan
las `TOP_N` películas que **no** calificó, ordenadas por $\hat r_{ij}$. `USUARIO` es el id tal como
aparece en MovieLens.

In [ ]:
def recomendar_top_n(U: np.ndarray, V: np.ndarray, M: np.ndarray, indice_usuario: int,
                     cantidad: int) -> list[tuple[str, float]]:
    "Las `cantidad` películas no calificadas por el usuario con mayor r̂ᵢⱼ."
    r_hat = predecir(U, V)[indice_usuario, :]
    candidatos = [j for j in range(V.shape[0]) if not M[indice_usuario, j]]
    candidatos.sort(key=lambda j: r_hat[j], reverse=True)
    return [(titulos_por_indice[j], float(r_hat[j])) for j in candidatos[:cantidad]]


if USUARIO not in datos.id_usuario_a_indice:
    raise UsuarioNoEncontradoError(USUARIO)
indice_usuario = datos.id_usuario_a_indice[USUARIO]

recomendaciones_als = recomendar_top_n(resultado_als.U, resultado_als.V, M, indice_usuario, TOP_N)
recomendaciones_gd = recomendar_top_n(resultado_gd.U, resultado_gd.V, M, indice_usuario, TOP_N)

print(f"Top-{TOP_N} para el usuario {USUARIO} (calificó {int(M[indice_usuario].sum())} películas)\n")
print(f"{'#':>3}  {'ALS':<58}{'Descenso de gradiente'}")
for posicion, ((titulo_als, r_als), (titulo_gd, r_gd)) in enumerate(zip(recomendaciones_als, recomendaciones_gd), 1):
    print(f"{posicion:>3}  {titulo_als[:50]:<50} {r_als:5.2f}  {titulo_gd[:50]:<50} {r_gd:5.2f}")

## 12. Qué aprendió cada factor latente

Las dimensiones latentes no se etiquetan ni se inicializan con ningún significado: emergen del
entrenamiento. Para mirarlas, se listan las películas con los valores más altos y más bajos en cada
columna de $V$ (la de ALS). La interpretación queda para el análisis.

In [ ]:
V_als = resultado_als.V
for factor in range(V_als.shape[1]):
    orden = np.argsort(V_als[:, factor])
    print(f"Factor {factor}")
    print("  valores más altos:")
    for j in orden[::-1][:5]:
        print(f"    {V_als[j, factor]:7.3f}  {titulos_por_indice[j]}")
    print("  valores más bajos:")
    for j in orden[:5]:
        print(f"    {V_als[j, factor]:7.3f}  {titulos_por_indice[j]}")
    print()

## 13. Probalo con tus propias calificaciones

Un usuario nuevo es una fila más de $R$. Con $V$ fija, su vector $u$ sale de **un paso de U de ALS**:
las mismas ecuaciones normales, así que se usa la misma función `resolver_factor`, ahora con una sola fila.

Abajo están las 30 películas más calificadas del conjunto filtrado. Editá el diccionario
`mis_calificaciones` con al menos 5 de ellas (notas de 1 a 5) y ejecutá la celda.

In [ ]:
MIN_CALIFICACIONES_NUEVO = 5
mas_calificadas = np.argsort(-M.sum(axis=0), kind="stable")[:30]
for j in mas_calificadas:
    print(titulos_por_indice[j])

In [ ]:
mis_calificaciones = {
    "Matrix, The (1999)": 5,
    "Toy Story (1995)": 4,
    "Independence Day (a.k.a. ID4) (1996)": 2,
    "Pulp Fiction (1994)": 5,
    "Forrest Gump (1994)": 3,
}

indice_por_titulo = {titulo: j for j, titulo in titulos_por_indice.items()}
if len(mis_calificaciones) < MIN_CALIFICACIONES_NUEVO:
    raise CalificacionesNuevasError(f"hacen falta al menos {MIN_CALIFICACIONES_NUEVO}, hay {len(mis_calificaciones)}")

r_nuevo = np.full((1, n), np.nan)
m_nuevo = np.zeros((1, n), dtype=bool)
for titulo, nota in mis_calificaciones.items():
    if titulo not in indice_por_titulo:
        raise CalificacionesNuevasError(f"'{titulo}' no está en el conjunto filtrado; copiá el título de la lista")
    if not 1 <= nota <= 5:
        raise CalificacionesNuevasError(f"la nota de '{titulo}' tiene que estar entre 1 y 5")
    r_nuevo[0, indice_por_titulo[titulo]] = nota
    m_nuevo[0, indice_por_titulo[titulo]] = True

# Paso de U para una sola fila, con la V de ALS fija
u_nuevo = resolver_factor(r_nuevo, m_nuevo, V_als)
r_hat_nuevo = predecir(u_nuevo, V_als)[0]

candidatos = [j for j in np.argsort(-r_hat_nuevo) if not m_nuevo[0, j]][:TOP_N]
print(f"u = {np.round(u_nuevo[0], 3)}\n")
print(f"Top-{TOP_N} para vos:")
for posicion, j in enumerate(candidatos, 1):
    print(f"{posicion:>3}. {titulos_por_indice[j]}  ({r_hat_nuevo[j]:.2f})")